# M5 IA Agêntica - Agente de Atendimento ao Cliente

## 1. Introdução

Como Andrew explicou na aula, *planejamento com execução de código* (planning with code execution) significa deixar o LLM **escrever código que se torna o próprio plano**.  
Comparado a planos baseados em texto simples ou JSON, essa abordagem é mais expressiva e flexível: o código não apenas documenta os passos, mas também pode executá-los diretamente.

Neste laboratório, você implementará esse padrão de design na prática.  
Em vez de pedir ao LLM para gerar um plano no formato JSON e então executar manualmente cada passo, permitiremos que ele **escreva código Python** que captura diretamente múltiplos passos de um plano. Ao executar esse código, podemos realizar consultas complexas automaticamente.  

Para tornar as coisas concretas, simulamos uma **loja de óculos de sol** com um **inventário** de produtos e um conjunto de **transações** (vendas, devoluções, atualizações de saldo). Este exemplo mostra como o LLM pode gerar código para consultar ou atualizar registros, demonstrando a flexibilidade desse padrão.

### 1.1 Visão Geral do Laboratório
Nós iremos:
1. Criar conjuntos de dados simples de **inventário** e **transações**.  
2. Construir um **bloco de esquema** (schema block) descrevendo os dados.  
3. Solicitar ao LLM que **escreva um plano como código Python** (com comentários explicando cada passo).  
4. Executar o código em um ambiente isolado (sandbox) para obter a resposta.  

### 1.2 Resultados de Aprendizado

Ao final deste laboratório, você será capaz de:

- **Explicar** por que deixar o modelo escrever código (em vez de planos JSON ou texto simples) permite um planejamento mais rico e flexível.  
- **Instruir (Prompt)** um LLM para produzir código Python com comentários passo a passo que documentam e executam o plano.  
- **Rodar** o código gerado com segurança em um sandbox e interpretar os resultados.  

Isso ilustra como *Code as Action* (Código como Ação) pode superar cadeias de ferramentas frágeis e abordagens de planejamento baseadas em JSON.

## 2. Configuração

In [ ]:
# ==== Imports ====
from __future__ import annotations
import json
from dotenv import load_dotenv
from openai import OpenAI
import re, io, sys, traceback, json
import os
from typing import Any, Dict, Optional
from tinydb import Query, where

# Utility modules
sys.path.append(os.path.abspath('../../'))
from utils import utils      # funções auxiliares para prompting/printing
from utils import inv_utils  # funções para inventário, transações, building schema e seeding TinyDB

load_dotenv()
client = OpenAI()

No módulo `inv_utils`, temos funções como:

- `create_inventory()` – constrói o inventário de óculos de sol.  
- `create_transactions()` – constrói o log de transações inicial.  
- `seed_db()` – carrega tanto o inventário quanto as transações em um armazenamento baseado em JSON.  
- `build_schema_block()` – gera uma descrição do esquema usada no prompt.  
- Auxiliares como `get_current_balance()` e `next_transaction_id()` – permitem que o LLM lide com atualizações consistentes entre inventário e transações.  

### 2.1 Criar Tabelas de Exemplo

Agora vamos criar duas pequenas tabelas para a simulação da loja de óculos, usando o **[TinyDB](https://tinydb.readthedocs.io/)** — um banco de dados orientado a documentos leve escrito em Python puro.  
O TinyDB armazena dados como documentos JSON e é adequado para pequenas aplicações ou protótipos, pois não requer configuração de servidor e permite consultar e atualizar dados facilmente.

As duas tabelas são:

- **`inventory_tbl`**: contém detalhes do produto como nome, ID do item, descrição, quantidade em estoque e preço.  
- **`transactions_tbl`**: começa com um saldo de abertura e posteriormente rastreará compras, devoluções e ajustes.  

Você gerará essas tabelas usando funções auxiliares em `inv_utils` e, em seguida, visualizará as primeiras linhas abaixo.

In [ ]:
db, inventory_tbl, transactions_tbl = inv_utils.seed_db()

Agora, você pode inspecionar os registros em cada tabela imprimindo-os como JSON formatado:

In [ ]:
utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table")
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table")

Como você pode ver acima, os esquemas de cada tabela são os seguintes:

<div style="border:1px solid #BFDBFE; border-left:6px solid #3B82F6; background:#EFF6FF; border-radius:6px; padding:16px; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif; line-height:1.6; color:#1E3A8A;">

  <h4 style="margin-top:0; color:#1E40AF;">Inventory Table (<code>inventory_tbl</code>)</h4>
  <ul>
    <li><strong>item_id</strong> (string): Identificador único do produto (ex: SG001).</li>
    <li><strong>name</strong> (string): Estilo dos óculos (ex: Aviator, Round).</li>
    <li><strong>description</strong> (string): Texto descritivo do produto.</li>
    <li><strong>quantity_in_stock</strong> (int): Estoque atual disponível.</li>
    <li><strong>price</strong> (float): Preço em USD.</li>
  </ul>
  <h4 style="margin-top:1em; color:#1E40AF;">Transactions Table (<code>transactions_tbl</code>)</h4>
  <ul>
    <li><strong>transaction_id</strong> (string): Identificador único (ex: TXN001).</li>
    <li><strong>customer_name</strong> (string): Nome do cliente, ou <code>OPENING_BALANCE</code> para entrada inicial.</li>
    <li><strong>transaction_summary</strong> (string): Breve descrição da transação.</li>
    <li><strong>transaction_amount</strong> (float): Quantia de dinheiro para esta transação.</li>
    <li><strong>balance_after_transaction</strong> (float): Saldo corrente após aplicar a transação.</li>
    <li><strong>timestamp</strong> (string): Data/hora formatada em ISO-8601 da transação.</li>
  </ul>
</div>


## Planejamento com Execução de Código

### 2.1. O Plano

Uma vez que o esquema esteja claro, você construirá o **prompt** que instrui o modelo a *planejar escrevendo código* e então executar esse código. Como Andrew enfatizou, o código é o plano: o modelo explica cada passo em comentários, depois o realiza. Seu prompt abaixo também faz com que o modelo decida por conta própria se a solicitação é somente leitura ou uma mudança de estado, e impõe execução segura (sem E/S, sem rede, apenas Query TinyDB, mutações consistentes).


In [ ]:
PROMPT = """You are a senior data assistant. PLAN BY WRITING PYTHON CODE USING TINYDB.

Database Schema & Samples (read-only):
{schema_block}

Execution Environment (already imported/provided):
- Variables: db, inventory_tbl, transactions_tbl  # TinyDB Table objects
- Helpers: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix="TXN") -> str
- Natural language: user_request: str  # the original user message

PLANNING RULES (critical):
- Derive ALL filters/parameters from user_request (shape/keywords, price ranges "under/over/between", stock mentions,
  quantities, buy/return intent). Do NOT hard-code values.
- Build TinyDB queries dynamically with Query(). If a constraint isn't in user_request, don't apply it.
- Be conservative: if intent is ambiguous, do read-only (DRY RUN).

TRANSACTION POLICY (hard):
- Do NOT create aggregated multi-item transactions.
- If the request contains multiple items, create a separate transaction row PER ITEM.
- For each item:
  - compute its own line total (unit_price * qty),
  - insert ONE transaction with that amount,
  - update balance sequentially (balance += line_total),
  - update the item’s stock.
- If any requested item lacks sufficient stock, do NOT mutate anything; reply with STATUS="insufficient_stock".

HUMAN RESPONSE REQUIREMENT (hard):
- You MUST set a variable named `answer_text` (type str) with a short, customer-friendly sentence (1–2 lines) in Portuguese.
- This sentence is the only user-facing message. No dataframes/JSON, no boilerplate disclaimers.
- If nothing matches, politely say so and offer a nearby alternative (closest style/price) or a next step.

ACTION POLICY:
- If the request clearly asks to change state (buy/purchase/return/restock/adjust):
    ACTION="mutate"; SHOULD_MUTATE=True; perform the change and write a matching transaction row.
  Otherwise:
    ACTION="read"; SHOULD_MUTATE=False; simulate and explain briefly as a dry run (in logs only).

FAILURE & EDGE-CASE HANDLING (must implement):
- Do not capture outer variables in Query.test. Pass them as explicit args.
- Always set a short `answer_text`. Also set a string `STATUS` to one of:
  "success", "no_match", "insufficient_stock", "invalid_request", "unsupported_intent".
- no_match: No items satisfy the filters → suggest the closest in style/price, or invite a different range.
- insufficient_stock: Item found but stock < requested qty → state available qty and offer the max you can fulfill.
- invalid_request: Unable to parse essential info (e.g., quantity for a purchase/return) → ask for the missing piece succinctly.
- unsupported_intent: The action is outside the store’s capabilities → provide the nearest supported alternative.
- In all cases, keep the tone helpful and concise (1–2 sentences). Put technical details (e.g., ACTION/DRY RUN) only in stdout logs.

OUTPUT CONTRACT:
- Return ONLY executable Python between these tags (no extra text):
  <execute_python>
  # your python
  </execute_python>

CODE CHECKLIST (follow in code):
1) Parse intent & constraints from user_request (regex ok).
2) Build TinyDB condition incrementally; query inventory_tbl.
3) If mutate: validate stock, update inventory, insert a transaction (new id, amount, balance, timestamp).
4) ALWAYS set:
   - `answer_text` (human sentence, required),
   - `STATUS` (see list above).
   Also print a brief log to stdout, e.g., "LOG: ACTION=read DRY_RUN=True STATUS=no_match".
5) Optional: set `answer_rows` or `answer_json` if useful, but `answer_text` is mandatory.

TONE EXAMPLES (for `answer_text`):
- success: "Sim, temos nossos óculos Classic, uma armação redonda, por $60."
- no_match: "Não temos armações redondas abaixo de $100 em estoque agora, mas nossa armação Moon redonda está disponível por $120."
- insufficient_stock: "Temos apenas 1 par de Classic restante; posso reservar para você."
- invalid_request: "Posso ajudar com isso—quantos pares você gostaria de comprar?"
- unsupported_intent: "Não podemos reformar armações, mas posso sugerir modelos novos similares."

Constraints:
- Use TinyDB Query for filtering. Standard library imports only if needed.
- Keep code clear and commented with numbered steps.

User request:
{question}
"""


### 2.2 Do Prompt ao Código (Planejamento no Código)

Vamos gerar código que **seja o plano**.

Em vez de pedir ao modelo que produza um plano em JSON e o execute passo a passo com muitas ferramentas pequenas, vamos fazer com que ele **escreva Python que codifique todo o plano** (ex: “filtrar isso, depois calcular aquilo, depois atualizar essa linha”). A função `generate_llm_code`:

1. **Constrói um esquema ao vivo** a partir de `inventory_tbl` e `transactions_tbl` para que o modelo veja campos reais, tipos e exemplos.
2. **Formata o prompt** com esse esquema mais a pergunta do usuário.
3. **Chama o modelo** para produzir uma resposta **plano-com-código** — tipicamente um bloco `<execute_python>...</execute_python>` cujo corpo contém a lógica passo a passo.
4. **Retorna a resposta completa** (incluindo o plano e o código).  
   *Nós não executamos nada nesta etapa.*

Por que esse padrão? Vamos aproveitar Python/TinyDB como uma caixa de ferramentas rica que o modelo já “conhece”, para que ele possa compor soluções de múltiplas etapas diretamente em código em vez de depender de um conjunto crescente de ferramentas personalizadas. Extrairemos e executaremos o código em uma etapa posterior.

In [ ]:
# ---------- 1) Code generation ----------
def generate_llm_code(
    prompt: str,
    *,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4o-mini",
    temperature: float = 0.2,
) -> str:
    """
    Ask the LLM to produce a plan-with-code response.
    Returns the FULL assistant content (including surrounding text and tags).
    The actual code extraction happens later in execute_generated_code.
    """
    schema_block = inv_utils.build_schema_block(inventory_tbl, transactions_tbl)
    prompt = PROMPT.format(schema_block=schema_block, question=prompt)

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": "You write safe, well-commented TinyDB code to handle data questions and updates."
            },
            {"role": "user", "content": prompt},
        ],
    )
    content = resp.choices[0].message.content or ""
    
    return content  

### 2.3 Testar um Prompt de Exemplo (Planning-in-Code)

Usaremos o mesmo prompt que Andrew usou na aula:

> **Prompt:** “Do you have any round sunglasses in stock that are under $100?” (Você tem óculos de sol redondos em estoque que custem menos de $100?)

Antes de gerar qualquer código, vamos inspecionar manualmente as tabelas TinyDB para ver se existem verdadeiramente armações *round* (correspondência apenas de palavra) e como são seus preços. Execute a próxima célula para visualizar o inventário e destacar itens que correspondem ao filtro de palavra “round”.

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Search the inventory table for documents where either the description OR the name
# contains the word "round" (case-insensitive). The check is done inline:
# - (v or "") ensures we handle None by converting it to an empty string
# - .lower() normalizes case
# - " round " enforces a crude word boundary (won't match "wraparound")
round_sunglasses = inventory_tbl.search(
    (Item.description.test(lambda v: " round " in ((v or "").lower()))) |
    (Item.name.test(        lambda v: " round " in ((v or "").lower())))
)

# Render the results as formatted JSON in the notebook UI
utils.print_html(json.dumps(round_sunglasses, indent=2), title="Inventory Status: Round Sunglasses")

Ótimo — nós temos armações redondas disponíveis. Pela nossa inspeção manual, há dois estilos redondos em estoque, mas apenas **um** está **abaixo de \$100**. Portanto, o item que satisfaz o requisito é:

````python
{
  "item_id": "SG005",
  "name": "Classic",
  "description": "Classic round profile with minimalist metal frames, offering a timeless and versatile style that fits both casual and formal wear.",
  "quantity_in_stock": 10,
  "price": 60
}
````

Agora vamos pedir ao modelo para **gerar um plano em código** que responda ao prompt do Andrew (sem execução ainda).

In [ ]:
# Andrew's prompt from the lecture
prompt_round = "Do you have any round sunglasses in stock that are under $100?"

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_round = generate_llm_code(
    prompt_round,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="gpt-4o-mini",
    temperature=1.0,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_round, title="Plan with Code (Full Response)")

### 2.4. Definir a função executora (rodar um plano dado)

Agora definiremos a função que **recebe um plano produzido pelo modelo e o executa** com segurança:

- Ela **aceita tanto** a resposta completa do LLM (com `<execute_python>…</execute_python>`) **quanto** código Python bruto.
- Ela **extrai** o bloco executável quando necessário.
- Ela roda o código em um **namespace controlado** (tabelas TinyDB + auxiliares seguros apenas).
- Ela captura **stdout**, **erros**, e as variáveis de resposta definidas pelo modelo (`answer_text`, `answer_rows`, `answer_json`).
- Ela renderiza snapshots de tabelas **antes/depois** para tornar os efeitos colaterais explícitos.

Este é o “executor” que transforma um **plano-como-código** em ações e uma resposta concisa para o usuário.


In [ ]:
# --- Helper: extract code between <execute_python>...</execute_python> ---
def _extract_execute_block(text: str) -> str:
    """
    Returns the Python code inside <execute_python>...</execute_python>.
    If no tags are found, assumes 'text' is already raw Python code.
    """
    if not text:
        raise RuntimeError("Empty content passed to code executor.")
    m = re.search(r"<execute_python>(.*?)</execute_python>", text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


# ---------- 2) Code execution ----------
def execute_generated_code(
    code_or_content: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    user_request: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Execute code in a controlled namespace.
    Accepts either raw Python code OR full content with <execute_python> tags.
    Returns minimal artifacts: stdout, error, and extracted answer.
    """
    # Extract code here (now centralized)
    code = _extract_execute_block(code_or_content)

    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": inv_utils.get_current_balance,
        "next_transaction_id": inv_utils.next_transaction_id,
        "user_request": user_request or "",
    }
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }

    # Capture stdout from the executed code
    _stdout_buf, _old_stdout = io.StringIO(), sys.stdout
    sys.stdout = _stdout_buf
    err_text = None
    try:
        exec(code, SAFE_GLOBALS, SAFE_LOCALS)
    except Exception:
        err_text = traceback.format_exc()
    finally:
        sys.stdout = _old_stdout
    printed = _stdout_buf.getvalue().strip()

    # Extract possible answers set by the generated code
    answer = (
        SAFE_LOCALS.get("answer_text")
        or SAFE_LOCALS.get("answer_rows")
        or SAFE_LOCALS.get("answer_json")
    )


    return {
        "code": code,            # <- ya sin etiquetas
        "stdout": printed,
        "error": err_text,
        "answer": answer,
        "transactions_tbl": transactions_tbl.all(),  # For inspection
        "inventory_tbl": inventory_tbl.all(),  # For inspection
    }

Você verificou as prateleiras e confirmou que há exatamente um estilo redondo abaixo de $100. Agora a parte divertida: vamos entregar o plano-como-código do modelo ao nosso executor e vê-lo trabalhar. O executor extrairá o bloco <code><execute_python>...</execute_python></code>, o executará em um sandbox bloqueado e, em seguida, mostrará tudo o que importa — o que mudou nas tabelas (antes/depois), quaisquer logs que o plano imprimiu e o `answer_text` final amigável ao cliente.

In [ ]:
# Execute the generated plan for the round-sunglasses question
result = execute_generated_code(
    full_content_round,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_round, # e.g., "Do you have any round sunglasses in stock that are under $100?"
)

# Peek at exactly what Python the plan executed
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

Como você pode ver, este é o resultado esperado com base em nossa análise manual anterior.

## 2.4 Devolver dois Óculos Aviator

No passo anterior, apenas **consultamos** os dados, então o inventário e as transações permaneceram inalterados.  
Agora vamos lidar com um cenário de **devolução** usando o padrão de planejamento em código:
> **Request:** “Return 2 Aviator sunglasses I bought last week.” (Devolver 2 óculos Aviator que comprei semana passada.)

Antes de gerar o plano, vamos **inspecionar o inventário atual** para o modelo *Aviator*.

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Query: fetch all inventory rows whose 'name' is exactly "Aviator".
# Notes:
# - This is a case-sensitive equality check. "aviator" won't match.
# - If you need case-insensitive matching, consider a .test(...) or .matches(...) with re.I.
aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

# Display the matched documents in a readable JSON panel
utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses before return")

O inventário confirma um SKU Aviator em estoque — **SG001 (Aviator)**: **23** unidades a **$80** cada. Agora vamos gerar um plano para responder ao prompt:

In [ ]:
prompt_aviator = "Return 2 Aviator sunglasses I bought last week."

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_aviator = generate_llm_code(
    prompt_aviator,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="gpt-4o-mini",
    temperature=1,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_aviator, title="Plan with Code (Full Response)")

Antes de executarmos o plano, vamos verificar o status atual das transações.

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table Before Return")

O log de transações mostra atualmente uma única entrada — o saldo inicial (`TXN001`) de `$500.00`. 

Pronto para ir — execute o plano rodando a célula abaixo.

In [ ]:
# Execute the generated plan for the round-sunglasses question
result = execute_generated_code(
    full_content_aviator,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_aviator, # e.g., "Return 2 aviator sunglasses I bought last week."
)

# Peek at exactly what Python the plan executed
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

Você pode ver abaixo que uma nova transação foi inserida para a devolução dos óculos Aviator.

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table After Return")

E ao rodar a célula abaixo, você verá o estoque de Aviator aumentar para 25 (`quantity_in_stock`).

In [ ]:
Item = Query()                  

aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses after return")

## 3. Juntando Tudo: Agente de Atendimento ao Cliente

Você construiu as peças — esquema, prompt, gerador de código e executor. Agora vamos conectá-las em um único auxiliar que recebe uma solicitação em linguagem natural, gera um plano-como-código, o executa com segurança e mostra o resultado (mais tabelas antes/depois).

**O que este agente faz**
- Opcionalmente re-semeia os dados de demonstração para uma execução limpa.
- Gera o plano (Python dentro de `<execute_python>…</execute_python>`).
- Executa o plano em um namespace controlado (TinyDB + auxiliares).
- Apresenta um `answer_text` conciso e renderiza snapshots antes/depois.

In [ ]:
def customer_service_agent(
    question: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4o-mini",
    temperature: float = 1.0,
    reseed: bool = False,
) -> dict:
    """
    End-to-end helper:
      1) (Optional) reseed inventory & transactions
      2) Generate plan-as-code from `question`
      3) Execute in a controlled namespace
      4) Render before/after snapshots and return artifacts

    Returns:
      {
        "full_content": <raw LLM response (may include <execute_python> tags)>,
        "exec": {
            "code": <extracted python>,
            "stdout": <plan logs>,
            "error": <traceback or None>,
            "answer": <answer_text/rows/json>,
            "inventory_after": [...],
            "transactions_after": [...]
        }
      }
    """
    # 0) Optional reseed
    if reseed:
        inv_utils.create_inventory()
        inv_utils.create_transactions()

    # 1) Show the question
    utils.print_html(question, title="User Question")

    # 2) Generate plan-as-code (FULL content)
    full_content = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        model=model,
        temperature=temperature,
    )
    utils.print_html(full_content, title="Plan with Code (Full Response)")

    # 3) Before snapshots
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · Before")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · Before")

    # 4) Execute
    exec_res = execute_generated_code(
        full_content,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question,
    )

    # 5) Render Results
    utils.print_html(exec_res["stdout"], title="Plan Execution · Logs (stdout)")
    if exec_res["error"]:
         utils.print_html(exec_res["error"], title="Plan Execution · Error")

    utils.print_html(exec_res["answer"], title="Plan Execution · Final Answer")
    
    # After snapshots
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · After")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · After")

    return {"full_content": full_content, "exec": exec_res}

### Exemplo 1: Uma compra complexa

Vamos tentar uma compra que desencadeie múltiplas atualizações de linha e deduza o saldo.

In [ ]:
question_buy = "I want to buy 1 Aviator and 2 Sport sunglasses."

out_buy = customer_service_agent(
    question_buy,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="gpt-4o-mini",
    reseed=True, # reset to clean state
)

### Exemplo 2: Consulta Ambígua (Dry Run)

Se o usuário for vago, o agente não deve mutar nada. O prompt o instrui a usar `ACTION='read'`, o que nosso executor respeita.

In [ ]:
question_vague = "I'm looking for something stylish."

out_vague = customer_service_agent(
    question_vague,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="gpt-4o-mini",
    reseed=False,
)

## 4. Conclusão

Neste laboratório, você foi além do planejamento JSON estático e dos tool loops (loops de ferramentas).
Ao pedir ao LLM para **planejar com código**, você permitiu que ele:

1. Expressasse lógica complexa (condicionais, loops) de forma nativa.
2. Realizasse consultas precisas filtradas em um banco de dados real.
3. Lidasse com mudanças de estado transacionais (estoque + atualização de saldo) em um bloco atômico.

Tudo isso aconteceu com segurança porque o executor roda o código gerado em um escopo restrito, com apenas ferramentas permitidas (Query, tabelas do DB) expostas.

Este padrão — **Code as Action** (Código como Ação) — é uma maneira poderosa de construir agentes autônomos que podem "pensar" em algoritmos em vez de apenas sequências de etapas.